# Tes Ektraksi Dokumen - Pipeline AI BRIDA

## Preparation

### Install Dependencies

In [1]:
!apt-get update
!apt-get install poppler-utils tesseract-ocr tesseract-ocr-ind
!pip install -r requirements.txt
!pip install --upgrade torch torchvision torchaudio
print("Setup siap!")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Fetched 3,917 B in 2s (1,610 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependen

### Import Library & Inisialisasi Model OCR

In [2]:
import os
import time
import pandas as pd
import zipfile
import shutil
import pymupdf
from paddleocr import PaddleOCR
from unstructured.partition.auto import partition

print("Memuat model PaddleOCR ke memori...")
ocr = PaddleOCR(use_textline_orientation=True, lang='id')
print("Model siap!")

Memuat model PaddleOCR ke memori...


/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/UVDoc`.


...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_medium_rec', None, None)
Using official model (PP-OCRv6_medium_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Model siap!


### Unzip Folder source_docs

In [3]:
path_zip = "source_docs.zip"

with zipfile.ZipFile(path_zip, 'r') as zip_ref:
    zip_ref.extractall(".")
    print("File berhasil diekstrak!")

File berhasil diekstrak!


## Konfigurasi Direktori & Fungsi Bantuan

In [4]:
# 1. KONFIGURASI PATH
FOLDER_DOKUMEN = "source_docs"
FOLDER_OUTPUT = "extracted_texts"
FILE_MAPPING = "mapping.csv"

# Buat folder output jika belum ada
os.makedirs(FOLDER_OUTPUT, exist_ok=True)

# 2. FUNGSI PEMBANTU
def cari_path_file(nama_file, folder_utama):
    """Mencari file secara otomatis hingga ke sub-folder."""
    for root, dirs, files in os.walk(folder_utama):
        if nama_file in files:
            return os.path.join(root, nama_file)
    return None

def is_native_pdf(file_path, min_char_per_page=50):
    if not file_path.lower().endswith('.pdf'):
        return False

    try:
        doc = pymupdf.open(file_path)
        if not doc: return False

        total_text = "".join([page.get_text().strip() for page in doc])
        avg_char_per_page = len(total_text) / len(doc) if len(doc) > 0 else 0
        return avg_char_per_page > min_char_per_page
    except Exception as e:
        print(f"[!] Gagal membaca metadata PDF: {e}")
        return False

print(f"Folder Output '{FOLDER_OUTPUT}' siap digunakan.")

Folder Output 'extracted_texts' siap digunakan.


## Fungsi Inti Smart Router

In [5]:
# 3. FUNGSI EKSTRAKSI (SMART ROUTER)
def run_paddleocr(path_dokumen):
    teks_hasil = ""
    try:
        hasil_ocr = ocr.predict(path_dokumen)
        if hasil_ocr:
            for halaman in hasil_ocr:
                if isinstance(halaman, dict) and 'rec_texts' in halaman:
                    for teks in halaman['rec_texts']:
                        teks_hasil += str(teks) + "\n"
                elif isinstance(halaman, list):
                    for item in halaman:
                        if isinstance(item, dict) and 'rec_texts' in item:
                            for teks in item['rec_texts']:
                                teks_hasil += str(teks) + "\n"

        if not teks_hasil.strip():
            teks_hasil = str(hasil_ocr)
    except Exception as e:
        return f"[ERROR PADDLEOCR]: {e}"

    return teks_hasil.strip()

def run_unstructuredio_auto(path_dokumen):
    try:
        elemen_auto = partition(
            filename=path_dokumen,
            strategy="auto",
            languages=["ind"],
            ocr_agent="unstructured.partition.utils.ocr_models.paddle_ocr.OCRAgentPaddle"
        )
        return "\n".join([str(elemen) for elemen in elemen_auto])
    except Exception as e:
        return f"[ERROR UNSTRUCTURED.IO]: {e}"

def extract_pdf_smart(file_path):
    if is_native_pdf(file_path):
        print("-> Rute: Native PDF (Unstructured Auto)")
        return run_unstructuredio_auto(file_path)
    else:
        print("-> Rute: Scanned/Gambar (PaddleOCR)")
        return run_paddleocr(file_path)

## Eksekusi Ekstraksi

In [6]:
# 4. EKSEKUSI PIPELINE EKSTRAKSI
print("\nMemulai Eksekusi Pipeline Ekstraksi...")
waktu_mulai = time.time()

if not os.path.exists(FILE_MAPPING):
    print(f"[!] ERROR FATAL: File {FILE_MAPPING} tidak ditemukan di Colab!")
else:
    df = pd.read_csv(FILE_MAPPING)
    total_file = len(df)

    for index, row in df.iterrows():
        doc_id = row['ID']
        nama_file = row['Nama_File']

        print(f"\n[{index+1}/{total_file}] Memproses: {nama_file}")

        path_dokumen = cari_path_file(nama_file, FOLDER_DOKUMEN)

        if not path_dokumen:
            print(f"[!] Gagal: File tidak ditemukan di dalam '{FOLDER_DOKUMEN}' atau sub-foldernya.")
            continue

        # Mulai Ekstraksi
        teks_hasil = extract_pdf_smart(path_dokumen)

        if not teks_hasil or not teks_hasil.strip():
            teks_hasil = "[INFO: Teks gagal diekstrak / dokumen kosong / file rusak]"

        # Simpan hasil
        nama_output = f"{doc_id}.txt"
        path_output = os.path.join(FOLDER_OUTPUT, nama_output)

        with open(path_output, "w", encoding="utf-8") as f:
            f.write(teks_hasil)

        print(f"[v] Selesai -> Disimpan di {path_output}")

    # 5. SELESAI
    durasi = time.time() - waktu_mulai
    jam, sisa = divmod(durasi, 3600)
    menit, detik = divmod(sisa, 60)

    print(f"\n" + "="*40)
    print(f"EKSTRAKSI SELESAI")
    print(f"Waktu total eksekusi: {int(jam)} jam {int(menit)} menit {detik:.2f} detik")
    print(f"Semua file teks telah tersimpan di folder '{FOLDER_OUTPUT}/'")
    print("="*40)


Memulai Eksekusi Pipeline Ekstraksi...

[1/42] Memproses: PERDA KOTA MAKASSAR TENTANG INOVASI DAERAH.pdf
-> Rute: Scanned/Gambar (PaddleOCR)
[v] Selesai -> Disimpan di extracted_texts/DOC-0001.txt

[2/42] Memproses: SK PENETAPAN INOVASI TAHUN 2025.pdf
-> Rute: Scanned/Gambar (PaddleOCR)


[v] Selesai -> Disimpan di extracted_texts/DOC-0002.txt

[3/42] Memproses: SK PENETAPAN INOVASI TAHUN 2026.pdf
-> Rute: Native PDF (Unstructured Auto)
[v] Selesai -> Disimpan di extracted_texts/DOC-0003.txt

[4/42] Memproses: Ketersediaan dan peran SDM terhadap inovasi berjumlah 32 orang tahun 2025.pdf
-> Rute: Scanned/Gambar (PaddleOCR)
[v] Selesai -> Disimpan di extracted_texts/DOC-0004.txt

[5/42] Memproses: Ketersediaan dan peran SDM terhadap inovasi berjumlah 33 orang tahun 2026.pdf
-> Rute: Native PDF (Unstructured Auto)
[v] Selesai -> Disimpan di extracted_texts/DOC-0005.txt

[6/42] Memproses: Dukungan Anggaran dialokasikan pada kegiatan penerapan inovasi di 2024.pdf
-> Rute: Scanned/Gambar (PaddleOCR)
[v] Selesai -> Disimpan di extracted_texts/DOC-0006.txt

[7/42] Memproses: Dukungan Anggaran dialokasikan pada kegiatan penerapan inovasi di 2025.pdf
-> Rute: Scanned/Gambar (PaddleOCR)
[v] Selesai -> Disimpan di extracted_texts/DOC-0007.txt

[8/42] Memproses: Dukungan Anggaran di

[v] Selesai -> Disimpan di extracted_texts/DOC-0020.txt

[21/42] Memproses: Sosialisasi Inovasi Daerah menggunakan Media Berita online tahun 2025.pdf
-> Rute: Native PDF (Unstructured Auto)


[v] Selesai -> Disimpan di extracted_texts/DOC-0021.txt

[22/42] Memproses: Sosialisasi Inovasi Daerah menggunakan Media Berita online tahun 2026.pdf
-> Rute: Native PDF (Unstructured Auto)
[v] Selesai -> Disimpan di extracted_texts/DOC-0022.txt

[23/42] Memproses: Manual Book Genting.pdf
-> Rute: Native PDF (Unstructured Auto)
[v] Selesai -> Disimpan di extracted_texts/DOC-0023.txt

[24/42] Memproses: Pedoman Teknis berupa buku yang dapat diakses secara online.pdf
-> Rute: Native PDF (Unstructured Auto)
[v] Selesai -> Disimpan di extracted_texts/DOC-0024.txt

[25/42] Memproses: Kemudahan Informasi Layanan melalui metode Hotline.pdf
-> Rute: Native PDF (Unstructured Auto)
[v] Selesai -> Disimpan di extracted_texts/DOC-0025.txt

[26/42] Memproses: Kemudahan Informasi Layanan melalui metode Manual.pdf
-> Rute: Native PDF (Unstructured Auto)
[v] Selesai -> Disimpan di extracted_texts/DOC-0026.txt

[27/42] Memproses: Kemudahan Informasi Layanan melalui metode Media Sosial.pdf
-> Rute: Nati

## Simpan output

In [7]:
nama_folder = 'extracted_texts'
nama_file_zip = 'extracted_texts'

# Proses kompresi folder menjadi .zip
shutil.make_archive(nama_file_zip, 'zip', nama_folder)

'/content/extracted_texts.zip'